In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False
# filtered_series_dict: 包含每只股票的 original, filtered, group, params

def build_dsp_factor(original, filtered, lookback=60):
    """
    构建DSP增强因子
    
    因子逻辑：
    - 分子：滤波后价格的动量（趋势强度）
    - 分母：噪声强度（残差的滚动波动率）
    - 因子值高 = 趋势强 + 噪声低 = 高质量买入信号
    
    参数:
    - original: 原始价格序列
    - filtered: 滤波后价格序列
    - lookback: 动量计算周期（60天）
    
    返回:
    - dsp_factor: DSP增强因子
    - filtered_momentum: 滤波后动量
    - original_momentum: 原始动量（对比用）
    - noise_intensity: 噪声强度
    """
    # 对齐数据
    common_idx = original.index.intersection(filtered.index)
    orig_aligned = original.loc[common_idx]
    filt_aligned = filtered.loc[common_idx]
    
    # 1. 滤波后动量（趋势强度）
    filtered_momentum = filt_aligned.pct_change(periods=lookback)
    
    # 2. 噪声强度（残差的滚动波动率）
    residual = orig_aligned.values - filt_aligned.values
    residual_series = pd.Series(residual, index=common_idx)
    noise_intensity = residual_series.rolling(window=lookback).std()
    
    # 3. DSP增强因子 = 动量 / 噪声强度
    dsp_factor = filtered_momentum / (noise_intensity + 1e-8)
    
    # 4. 原始动量（作为对比基准）
    original_momentum = orig_aligned.pct_change(periods=lookback)
    
    return {
        'dsp_factor': dsp_factor,
        'filtered_momentum': filtered_momentum,
        'original_momentum': original_momentum,
        'noise_intensity': noise_intensity
    }

# 1. 读取总表
df_all = pd.read_csv(r"..\数据\全股票滤波结果总表.csv")

# 2. 把日期设为索引
df_all['trade_date'] = pd.to_datetime(df_all['trade_date'])
df_all = df_all.set_index('trade_date').sort_index()

# 3. 按股票代码分组遍历
factors_dict = {}
factor_rows = []  # 用于保存总表
for code, df_stock in df_all.groupby('code'):
    # 取出当前股票：原始收盘价 + 滤波后价格
    original = df_stock['close_price']    # 正确字段名
    filtered = df_stock['filtered_price'] # 正确字段名
    
    # 构建DSP因子
    factors = build_dsp_factor(original, filtered, lookback=60)
    factors_dict[code] = factors
    
     # 拼合成一行行数据，方便存 CSV
    temp = pd.DataFrame({
        'code': code,
        'close_price': original,
        'filtered_price': filtered,
        'dsp_factor': factors['dsp_factor'],
        'filtered_momentum': factors['filtered_momentum'],
        'original_momentum': factors['original_momentum'],
        'noise_intensity': factors['noise_intensity']
    })
    factor_rows.append(temp)

    # 输出统计信息
    dsp_factor = factors['dsp_factor'].dropna()
    if len(dsp_factor) > 0:
        print(f"股票 {code} | DSP因子均值={dsp_factor.mean():8.4f} | 标准差={dsp_factor.std():8.4f}")

# 3. 保存完整因子总表
df_factor_all = pd.concat(factor_rows)
save_path = r"..\数据\DSP因子总表.csv"
df_factor_all.to_csv(save_path, encoding='utf-8-sig')
print("\n✅ 因子总表已保存至：")
print(save_path)

# 4. 批量画图：价格 + DSP因子 + 噪声强度
print("\n开始生成因子对比图...")
out_dir = Path(r"..\图表\因子图表")
out_dir.mkdir(exist_ok=True)

for code, df_stock in df_all.groupby('code'):
    factors = factors_dict[code]

    fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
    ax1, ax2, ax3 = axes

    # 子图1：原始价 & 滤波价
    ax1.plot(df_stock.index, df_stock['close_price'], label='原始收盘价', c='#1f77b4', linewidth=1.2)
    ax1.plot(df_stock.index, df_stock['filtered_price'], label='滤波趋势', c='#ff7f0e', linewidth=2)
    ax1.set_title(f'{code} 价格与滤波趋势', fontsize=14)
    ax1.legend()
    ax1.grid(alpha=0.3)

    # 子图2：DSP增强因子
    ax2.plot(factors['dsp_factor'], label='DSP因子', c='#2ca02c', linewidth=1.5)
    ax2.axhline(0, c='red', linestyle='--', alpha=0.7)
    ax2.set_title('DSP 增强因子（趋势/噪声）')
    ax2.legend()
    ax2.grid(alpha=0.3)

    # 子图3：噪声强度
    ax3.plot(factors['noise_intensity'], label='噪声强度', c='#d62728', linewidth=1.2)
    ax3.set_title('残差滚动波动率（噪声强度）')
    ax3.legend()
    ax3.grid(alpha=0.3)

    plt.tight_layout()
    fig.savefig(out_dir / f"因子图表_{code}.png", dpi=150, bbox_inches='tight')
    plt.close(fig)

print("="*60)
print("✅ 全部完成！")
print("  - DSP因子总表已保存")
print("  - 每只股票三张对比图已生成")
print("="*60)

股票 000333.SZ | DSP因子均值=  0.0387 | 标准差=  0.1095
股票 002475.SZ | DSP因子均值=  0.0645 | 标准差=  0.1644
股票 002594.SZ | DSP因子均值=  0.0007 | 标准差=  0.0215
股票 300059.SZ | DSP因子均值= -0.1094 | 标准差=  0.4428
股票 300308.SZ | DSP因子均值=  0.0358 | 标准差=  0.0946
股票 300502.SZ | DSP因子均值=  0.0272 | 标准差=  0.0593
股票 300750.SZ | DSP因子均值=  0.0027 | 标准差=  0.0340
股票 600030.SH | DSP因子均值=  0.0036 | 标准差=  0.2840
股票 600036.SH | DSP因子均值=  0.0242 | 标准差=  0.2392
股票 600276.SH | DSP因子均值=  0.0210 | 标准差=  0.1174
股票 600519.SH | DSP因子均值= -0.0014 | 标准差=  0.0039
股票 600900.SH | DSP因子均值=  0.1190 | 标准差=  0.3667
股票 601166.SH | DSP因子均值=  0.0571 | 标准差=  0.4509
股票 601318.SH | DSP因子均值=  0.0239 | 标准差=  0.1465
股票 601398.SH | DSP因子均值=  0.8363 | 标准差=  1.1138
股票 601899.SH | DSP因子均值=  0.2303 | 标准差=  0.4090
股票 603259.SH | DSP因子均值=  0.0215 | 标准差=  0.1073
股票 688041.SH | DSP因子均值=  0.0225 | 标准差=  0.0614
股票 688256.SH | DSP因子均值=  0.0080 | 标准差=  0.0229
股票 688981.SH | DSP因子均值=  0.0106 | 标准差=  0.0829

✅ 因子总表已保存至：
..\数据\DSP因子总表.csv

开始生成因子对比图...
✅ 全部完成！
  - DSP